In [ ]:
# KALMAN Historical Quant 2017 -> Now / Colab One-Cell
# Research / BACKTEST / SHADOW only. No Toss orders. No Neon writes.

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, json, shutil, subprocess, textwrap
from pathlib import Path

# =========================
# USER CONFIG
# =========================
START_DATE = "2017-01-01"
FORCE_REBUILD_MATRICES = False
ENABLE_QLIB_RECORDER = False   # Optional; slower and not required for performance comparison
RUN_RISKFOLIO = True

TRAIN_OBS = 504
VALID_OBS = 63
TEST_OBS = 126
MAX_HOLD_BARS = 20

PORTFOLIO_METHOD = "hrp"
PORTFOLIO_LOOKBACK = 180
PORTFOLIO_MIN_OBS = 90
PORTFOLIO_REBALANCE = "M"

MYDRIVE = Path("/content/drive/MyDrive")
REPO_ROOT = Path("/content/Codex")
RISK_VENV = Path("/content/.venv-riskfolio")

def run(cmd, *, cwd=None, env=None):
    printable = " ".join(map(str, cmd))
    print("\n$", printable)
    try:
        subprocess.run([str(x) for x in cmd], check=True, cwd=cwd, env=env)
    except subprocess.CalledProcessError as exc:
        print("\n[COMMAND FAILED]")
        print("return code:", exc.returncode)
        print("command    :", printable)
        raise

# =========================
# 1) REPO
# =========================
if not (REPO_ROOT / ".git").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/kimtk94/Codex.git", str(REPO_ROOT)])
else:
    run(["git", "-C", str(REPO_ROOT), "fetch", "origin"])
    run(["git", "-C", str(REPO_ROOT), "checkout", "main"])
    run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", "main"])

APP_ROOT = REPO_ROOT / "kalman-toss-gateway"
assert APP_ROOT.exists(), APP_ROOT
os.chdir(APP_ROOT)
sys.path.insert(0, str(APP_ROOT))

head = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repo HEAD:", head)

# =========================
# 2) ISOLATED COLAB ENVIRONMENT
# =========================
# Do NOT upgrade/downgrade Colab's already-loaded NumPy/SciPy stack.
# All Kalman research code runs in an isolated virtualenv.
run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])

if not (KALMAN_VENV / "bin/python").exists():
    run([sys.executable, "-m", "virtualenv", str(KALMAN_VENV)])

KPY = KALMAN_VENV / "bin/python"
KPIP = KALMAN_VENV / "bin/pip"

run([
    str(KPIP), "install", "-q",
    "pandas", "numpy", "scikit-learn", "pyarrow", "python-dotenv",
    "-r", str(APP_ROOT / "research/quant_stack/requirements-pypfopt.txt")
])

if ENABLE_QLIB_RECORDER:
    run([
        str(KPIP), "install", "-q",
        "-r", str(APP_ROOT / "research/quant_stack/requirements-qlib.txt")
    ])

run([
    str(KPY), "-c",
    "import numpy,pandas,scipy,sklearn,pyarrow,pypfopt;"
    "print('KALMAN ENV OK',"
    "'numpy='+numpy.__version__,"
    "'pandas='+pandas.__version__,"
    "'scipy='+scipy.__version__,"
    "'sklearn='+sklearn.__version__)"
])

# Colab itself already includes pandas/numpy; use those only for display/audit.
import pandas as pd
import numpy as np

# =========================
# 3) FIND DRIVE DATA ROOT
# =========================
def find_model_root():
    candidates = [
        MYDRIVE / "Market_Model_V2",
        MYDRIVE / "Kalman" / "Market_Model_V2",
        MYDRIVE / "kalman" / "Market_Model_V2",
    ]
    for p in candidates:
        if p.exists():
            return p

    # Bounded search to avoid walking the entire Drive forever.
    base_depth = len(MYDRIVE.parts)
    for root, dirs, files in os.walk(MYDRIVE):
        root_p = Path(root)
        depth = len(root_p.parts) - base_depth
        if depth > 4:
            dirs[:] = []
            continue
        if root_p.name == "Market_Model_V2":
            return root_p
    return None

MODEL_ROOT = find_model_root()
if MODEL_ROOT is None:
    raise FileNotFoundError(
        "Google Drive에서 Market_Model_V2 폴더를 찾지 못했습니다. "
        "MyDrive 루트 또는 Kalman 하위에 Market_Model_V2가 있어야 합니다."
    )

DATA_ROOT = MODEL_ROOT.parent
MARKET_ROOT = DATA_ROOT / "Market_Data" / "v2"
FEATURE_ROOT = DATA_ROOT / "Market_Features" / "v2"
MATRIX_DIR = MODEL_ROOT / "matrices"
OUTPUT_DIR = MODEL_ROOT / "historical_quant_v1_colab"

SPEC = APP_ROOT / "config/model-v2-spec.json"
UNIVERSE = APP_ROOT / "config/market-data-v2-universe.json"

print("\nDrive paths")
print("DATA_ROOT   :", DATA_ROOT)
print("MODEL_ROOT  :", MODEL_ROOT)
print("MARKET_ROOT :", MARKET_ROOT)
print("FEATURE_ROOT:", FEATURE_ROOT)
print("MATRIX_DIR  :", MATRIX_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)

# =========================
# 4) BUILD MATRICES IF NEEDED
# =========================
required_matrix_files = []
for _market in ("us", "kr", "btc"):
    required_matrix_files.extend([
        MATRIX_DIR / f"{_market}_matrix.parquet",
        MATRIX_DIR / f"{_market}_matrix_manifest.json",
    ])
need_build = FORCE_REBUILD_MATRICES or not all(p.exists() for p in required_matrix_files)

if need_build:
    if not MARKET_ROOT.exists():
        raise FileNotFoundError(f"Market data root missing: {MARKET_ROOT}")
    if not FEATURE_ROOT.exists():
        raise FileNotFoundError(f"Feature root missing: {FEATURE_ROOT}")

    print("\nBuilding fresh Model V2 feature matrices in Colab...")
    MATRIX_DIR.mkdir(parents=True, exist_ok=True)
    run([
        str(KPY), "-m", "research.model_v2.build_feature_matrix",
        "--market-root", str(MARKET_ROOT),
        "--feature-root", str(FEATURE_ROOT),
        "--universe", str(UNIVERSE),
        "--spec", str(SPEC),
        "--output-dir", str(MATRIX_DIR),
    ], cwd=APP_ROOT)
else:
    print("\nExisting matrices found; reusing them.")

# =========================
# 5) NORMALIZE SERVER MANIFEST PATHS FOR COLAB
# =========================
MIRROR_DIR = Path("/content/kalman_matrix_mirror")
if MIRROR_DIR.exists():
    shutil.rmtree(MIRROR_DIR)
MIRROR_DIR.mkdir(parents=True)

def remap_server_path(raw):
    raw = str(raw)
    p = Path(raw)
    if p.exists():
        return raw

    replacements = []
    if raw.startswith("/mnt/gdrive/"):
        replacements.append(MYDRIVE / raw[len("/mnt/gdrive/"):])

    for token in ("Market_Data/", "Market_Features/", "Market_Model_V2/"):
        if token in raw:
            suffix = raw.split(token, 1)[1]
            replacements.append(DATA_ROOT / token.rstrip("/") / suffix)
            replacements.append(MYDRIVE / token.rstrip("/") / suffix)

    for candidate in replacements:
        if candidate.exists():
            return str(candidate)
    return raw

# If old server-generated manifests cannot resolve their anchor OHLC files in
# Colab, rebuild matrices once so manifests point to Drive-local raw files.
spec_payload = json.loads(SPEC.read_text(encoding="utf-8"))

def unresolved_anchor_markets():
    broken = []
    for market in ("us", "kr", "btc"):
        manifest = MATRIX_DIR / f"{market}_matrix_manifest.json"
        if not manifest.exists():
            broken.append(market)
            continue
        payload = json.loads(manifest.read_text(encoding="utf-8"))
        anchor_key = spec_payload["markets"][market.upper()]["anchor_key"]
        found = False
        for old_path, meta in payload.get("input_files", {}).items():
            if str(meta.get("kind")) != "raw":
                continue
            if str(meta.get("fetch_key")) != anchor_key:
                continue
            mapped = remap_server_path(old_path)
            if Path(mapped).exists():
                found = True
                break
        if not found:
            broken.append(market)
    return broken

broken_anchors = unresolved_anchor_markets()
if broken_anchors:
    print("\nBroken/unresolved anchor manifests:", broken_anchors)
    if MARKET_ROOT.exists() and FEATURE_ROOT.exists():
        print("Rebuilding matrices to refresh Drive-local raw paths...")
        MATRIX_DIR.mkdir(parents=True, exist_ok=True)
        run([
            str(KPY), "-m", "research.model_v2.build_feature_matrix",
            "--market-root", str(MARKET_ROOT),
            "--feature-root", str(FEATURE_ROOT),
            "--universe", str(UNIVERSE),
            "--spec", str(SPEC),
            "--output-dir", str(MATRIX_DIR),
        ], cwd=APP_ROOT)
    else:
        raise FileNotFoundError(
            "Anchor raw OHLC paths in matrix manifests cannot be resolved, and "
            f"Market_Data/Market_Features roots are unavailable. Broken={broken_anchors}"
        )

for market in ("us", "kr", "btc"):
    parquet = MATRIX_DIR / f"{market}_matrix.parquet"
    manifest = MATRIX_DIR / f"{market}_matrix_manifest.json"

    if not parquet.exists() or not manifest.exists():
        raise FileNotFoundError(
            f"Matrix artifacts missing for {market}: {parquet} / {manifest}"
        )

    shutil.copy2(parquet, MIRROR_DIR / parquet.name)

    payload = json.loads(manifest.read_text(encoding="utf-8"))
    input_files = payload.get("input_files", {})
    remapped = {}
    changes = 0
    for old_path, meta in input_files.items():
        new_path = remap_server_path(old_path)
        if new_path != old_path:
            changes += 1
        remapped[new_path] = meta
    payload["input_files"] = remapped

    anchor_key = spec_payload["markets"][market.upper()]["anchor_key"]
    anchor_candidates = [
        path for path, meta in remapped.items()
        if str(meta.get("kind")) == "raw"
        and str(meta.get("fetch_key")) == anchor_key
    ]
    if not anchor_candidates or not any(Path(path).exists() for path in anchor_candidates):
        raise FileNotFoundError(
            f"{market.upper()} anchor raw OHLC unresolved after remap: "
            f"anchor_key={anchor_key}, candidates={anchor_candidates}"
        )

    (MIRROR_DIR / manifest.name).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    print(f"{market.upper()} manifest remapped paths: {changes}")

# =========================
# 6) COVERAGE AUDIT
# =========================
print("\n" + "=" * 78)
print("MATRIX COVERAGE")
print("=" * 78)

coverage = []
for market in ("us", "kr", "btc"):
    df = pd.read_parquet(MIRROR_DIR / f"{market}_matrix.parquet", columns=["as_of"])
    ts = pd.to_datetime(df["as_of"], utc=True, errors="coerce").dropna()
    row = {
        "market": market.upper(),
        "rows": len(df),
        "min_as_of": ts.min(),
        "max_as_of": ts.max(),
    }
    coverage.append(row)
    print(
        f"{row['market']:4s} rows={row['rows']:,} "
        f"min={row['min_as_of']} max={row['max_as_of']}"
    )

coverage_df = pd.DataFrame(coverage)
earliest = pd.to_datetime(coverage_df["min_as_of"], utc=True, errors="coerce").min()
if pd.notna(earliest) and earliest > pd.Timestamp("2018-01-01", tz="UTC"):
    print("\n[WARN] 현재 feature matrix가 2017까지 내려가지 않습니다.")
    print("       실행은 계속하지만, 진짜 2017 backtest가 되려면 raw/feature history backfill이 추가로 필요합니다.")

# =========================
# 7) RUN HISTORICAL WALK-FORWARD + HRP
# =========================
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("\nFresh run output:", OUTPUT_DIR)

args = [
    str(KPY), "-m", "research.quant_stack.experiment_runner",
    "--matrix-dir", str(MIRROR_DIR),
    "--spec", str(SPEC),
    "--output-dir", str(OUTPUT_DIR),
    "--start-date", START_DATE,
    "--train", str(TRAIN_OBS),
    "--valid", str(VALID_OBS),
    "--test", str(TEST_OBS),
    "--max-hold-bars", str(MAX_HOLD_BARS),
    "--git-sha", head,
    "--portfolio-targets",
    "--portfolio-method", PORTFOLIO_METHOD,
    "--portfolio-lookback-days", str(PORTFOLIO_LOOKBACK),
    "--portfolio-min-observations", str(PORTFOLIO_MIN_OBS),
    "--portfolio-rebalance", PORTFOLIO_REBALANCE,
]

if ENABLE_QLIB_RECORDER:
    qlib_tracking = MODEL_ROOT / "qlib_mlruns_colab"
    qlib_provider = Path("/content/qlib_provider")
    args += [
        "--qlib-recorder",
        "--qlib-tracking-root", str(qlib_tracking),
        "--qlib-provider-root", str(qlib_provider),
        "--qlib-experiment-name", "kalman_historical_quant_colab",
    ]

run(args, cwd=APP_ROOT)

# =========================
# 8) VALIDATION GATE
# =========================
run([
    str(KPY), "-m", "research.quant_stack.validate_artifacts",
    "--output-dir", str(OUTPUT_DIR),
], cwd=APP_ROOT)

# =========================
# 9) LEAN-INSPIRED SHADOW EXECUTION
# =========================
run([
    str(KPY), "-m", "research.quant_stack.lean_execution_runner",
    "--output-dir", str(OUTPUT_DIR),
    "--max-symbol-weight", "0.75",
    "--max-gross-weight", "1.0",
    "--min-order-notional", "10",
    "--max-single-order-fraction", "0.80",
    "--max-total-turnover-fraction", "2.0",
], cwd=APP_ROOT)

# =========================
# 10) ISOLATED RISKFOLIO BENCHMARK
# =========================
if RUN_RISKFOLIO:
    if not (RISK_VENV / "bin/python").exists():
        run([sys.executable, "-m", "virtualenv", str(RISK_VENV)])

    risk_py = RISK_VENV / "bin/python"
    risk_pip = RISK_VENV / "bin/pip"

    try:
        subprocess.run(
            [str(risk_py), "-c", "import riskfolio"],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        print("\n[OK] Riskfolio already installed")
    except subprocess.CalledProcessError:
        run([str(risk_pip), "install", "-q", "--upgrade", "pip"])
        run([
            str(risk_pip), "install", "-q",
            "-r", str(APP_ROOT / "research/quant_stack/requirements-riskfolio.txt")
        ])

    risk_env = os.environ.copy()
    risk_env["PYTHONPATH"] = str(APP_ROOT)

    run([
        str(risk_py), "-m", "research.quant_stack.riskfolio_benchmark_runner",
        "--output-dir", str(OUTPUT_DIR),
        "--lookback-days", str(PORTFOLIO_LOOKBACK),
        "--min-observations", str(PORTFOLIO_MIN_OBS),
        "--rebalance", PORTFOLIO_REBALANCE,
    ], cwd=APP_ROOT, env=risk_env)

# =========================
# 11) FINAL SUMMARY
# =========================
def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

def pct(x):
    try:
        return f"{float(x)*100:.2f}%"
    except Exception:
        return "-"

def num(x):
    try:
        return f"{float(x):.3f}"
    except Exception:
        return "-"

print("\n" + "=" * 88)
print("KALMAN HISTORICAL QUANT FINAL SUMMARY")
print("=" * 88)

rows = []
for market in ("us", "kr", "btc"):
    perf = load_json(OUTPUT_DIR / market / "historical_performance.json")
    if not perf:
        continue
    rows.append({
        "method": market.upper(),
        "source": "MARKET_BACKTEST",
        "total_return": perf.get("total_return"),
        "cagr": perf.get("cagr"),
        "sharpe": perf.get("sharpe"),
        "max_drawdown": perf.get("max_drawdown"),
        "trade_count": perf.get("trade_count"),
    })

hrp = load_json(OUTPUT_DIR / "portfolio" / "portfolio_performance.json")
if hrp:
    rows.append({
        "method": "HRP",
        "source": "PYPFOPT",
        "total_return": hrp.get("total_return"),
        "cagr": hrp.get("cagr"),
        "sharpe": hrp.get("sharpe"),
        "max_drawdown": hrp.get("max_drawdown"),
        "trade_count": None,
    })

risk_cmp = OUTPUT_DIR / "riskfolio" / "riskfolio_comparison.csv"
if risk_cmp.exists():
    rdf = pd.read_csv(risk_cmp)
    for _, r in rdf.iterrows():
        if str(r.get("source", "")).upper() == "PYPFOPT":
            continue
        rows.append({
            "method": str(r.get("method")),
            "source": str(r.get("source", "RISKFOLIO")),
            "total_return": r.get("total_return"),
            "cagr": r.get("cagr"),
            "sharpe": r.get("sharpe"),
            "max_drawdown": r.get("max_drawdown"),
            "trade_count": None,
        })

summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    raw_summary = summary_df.copy()
    for c in ("total_return", "cagr", "max_drawdown"):
        summary_df[c] = pd.to_numeric(summary_df[c], errors="coerce").map(
            lambda x: f"{x*100:.2f}%" if pd.notna(x) else "-"
        )
    summary_df["sharpe"] = pd.to_numeric(summary_df["sharpe"], errors="coerce").round(3)
    display(summary_df)

    portfolio_only = raw_summary[
        raw_summary["source"].isin(["PYPFOPT", "RISKFOLIO"])
    ].copy()
    portfolio_only["sharpe"] = pd.to_numeric(portfolio_only["sharpe"], errors="coerce")
    portfolio_only = portfolio_only.dropna(subset=["sharpe"])
    if not portfolio_only.empty:
        best = portfolio_only.loc[portfolio_only["sharpe"].idxmax()]
        print(f"\nBEST PORTFOLIO SHARPE: {best['method']} / {best['source']} / {best['sharpe']:.3f}")

    raw_summary.to_csv(OUTPUT_DIR / "kalman_final_comparison.csv", index=False)

execution = load_json(OUTPUT_DIR / "execution" / "execution_status.json")
print("\nLEAN-INSPIRED SHADOW EXECUTION")
print(json.dumps(execution, ensure_ascii=False, indent=2, default=str))

validation = load_json(OUTPUT_DIR / "historical_validation_report.json")
print("\nVALIDATION")
print(json.dumps(validation, ensure_ascii=False, indent=2, default=str))

print("\nOUTPUT:", OUTPUT_DIR)
print("Comparison CSV:", OUTPUT_DIR / "kalman_final_comparison.csv")
print("\nSAFETY: LIVE=False / Toss=False / Neon write=False")
